<a href="https://colab.research.google.com/github/sakshi987123/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sakshi987123/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb pandas pyarrow huggingface_hub

Load your Hugging Face token

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "HF_TOKEN not found.")

Token loaded successfully!


Connect DuckDB

In [3]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully!")

DuckDB connected successfully!


Load the HTTPFS extension

In [4]:

con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

print("Extensions loaded!")

Extensions loaded!


Connect to the FlyRank warehouse

In [5]:
con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [6]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [7]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_query_90d']}
LIMIT 5;
""").df()

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


## 1. Question

*The research question and the decision it supports.*

# Predicting SEO Content Optimization Opportunities Using Machine Learning

## Research Question

Can machine learning models accurately identify website content that should be prioritized for SEO optimization using historical Google Search Console (GSC) and Google Analytics (GA4) data?

# Abstract

This study investigates whether machine learning can identify website content that should be prioritized for SEO optimization using historical Google Search Console (GSC) and Google Analytics (GA4) data. A rule-based baseline was compared with Logistic Regression, Decision Tree, and Random Forest models using the FlyRank ML Internship warehouse dataset. Model performance was evaluated using Accuracy, Precision, Recall, and F1 Score together with validation and leakage audits. Random Forest achieved the strongest observed performance on the evaluation dataset. The resulting action playbook is intended to support SEO analysts in making informed optimization decisions while maintaining honest and public-safe reporting.

## Objective

The objective of this project is to develop and evaluate machine learning models that assist SEO analysts in identifying web pages requiring optimization. The project compares a rule-based baseline with Logistic Regression, Decision Tree, and Random Forest models using anonymized FlyRank search analytics data. The study aims to generate a ranked action playbook that supports human decision-making while following honest validation practices, leakage checks, and public-safe reporting.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

# Data

## Dataset

This project uses the **FlyRank ML Internship Warehouse Dataset**, which contains anonymized Google Search Console (GSC) and Google Analytics 4 (GA4) data for SEO analysis. The dataset was designed for educational and research purposes while preserving user and client privacy.

## Dataset Release

- **Dataset:** FlyRank ML Internship Warehouse
- **Primary Tables Used:**
  - `fact_daily`
  - `fact_query_90d`

## Time Window

This study uses data from **March 2026 (2026-03)** for feature engineering, model training, and evaluation. A mid-panel month was selected to avoid using the final month of the dataset for model development.

## Data Used

The following information was used to build the machine learning models:

- Google Search Console impressions
- Google Search Console clicks
- Click-Through Rate (CTR)
- Average search position
- Google Analytics page views
- Sessions
- Engaged sessions

## Data Excluded

To ensure privacy and maintain public-safe reporting, the following information was excluded:

- Client names
- Website URLs
- Search queries
- Private identifiers
- Any confidential business information

Only anonymized and aggregated metrics were used throughout this project.

## Public-Safe Data Usage

The analysis was performed entirely on anonymized warehouse data provided through the FlyRank ML Internship. No private client information, URLs, or search queries are included in this report.

In [8]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

# Methodology

## Overview

This project follows a structured machine learning workflow to identify website content that should be prioritized for SEO optimization. The workflow includes data preparation, feature engineering, baseline development, model training, validation, leakage analysis, and recommendation generation.

## Features Used

The following historical features were used as model inputs:

- Google Search Console (GSC) Impressions
- Google Search Console (GSC) Clicks
- Click-Through Rate (CTR)
- Average Search Position
- Google Analytics (GA4) Page Views
- GA4 Sessions
- GA4 Engaged Sessions

These features were selected because they represent historical search performance and user engagement available before making optimization decisions.

## Label Definition

The target variable was created using a rule-based definition:

- Target = 1 (Needs Optimization): Pages with high impressions (≥1000) and low CTR (<1%)
- Target = 0 (No Immediate Action): All other pages

This label was used to train and evaluate the classification models.

## Baseline

A rule-based baseline was developed before training machine learning models. The baseline prioritizes pages with high impressions but low click-through rates and assigns recommendation scores based on predefined thresholds.

## Machine Learning Models

Three supervised learning algorithms were evaluated:

- Logistic Regression
- Decision Tree
- Random Forest

The models were trained using the selected features and compared using common classification metrics.

## Validation Design

An 80:20 train-test split was initially used for model evaluation. To obtain a more realistic estimate of performance, grouped validation based on client identifiers was also considered to reduce information sharing between training and testing data.

## Leakage Checks

Potential label-derived leakage was reviewed because the target variable was constructed using impressions and CTR. The project included a leakage audit to identify features that might directly influence the target definition. Results were interpreted carefully to avoid overstating model performance.

## Evaluation Metrics

The models were evaluated using:

- Accuracy
- Precision
- Recall
- F1 Score

Random Forest achieved the strongest observed performance among the evaluated models and was selected as the final model for generating ranked recommendations.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

# Results

## Model Performance

Three machine learning models were trained and evaluated to identify web pages requiring SEO optimization. Their performance was compared with the rule-based baseline using the same dataset and evaluation strategy.

### Model Comparison

| Model | Accuracy | Precision | Recall | F1 Score |
|--------|---------:|----------:|--------:|---------:|
| Rule-Based Baseline | Rule-based | - | - | - |
| Logistic Regression | 99.91% | 95.86% | 92.67% | 94.24% |
| Decision Tree | 100.00% | 100.00% | 100.00% | 100.00% |
| Random Forest | 100.00% | 100.00% | 100.00% | 100.00% |

## Best Performing Model

Among the evaluated models, **Random Forest** achieved the strongest observed performance while also providing feature importance scores. It was selected as the final model for generating ranked SEO recommendations because it combines multiple decision trees, reduces overfitting compared to a single tree, and offers better interpretability through feature importance analysis.

## Feature Importance

The Random Forest model identified the following features as the most influential:

1. Google Search Console Impressions
2. Click-Through Rate (CTR)
3. Google Search Console Clicks
4. Average Search Position
5. GA4 Page Views
6. GA4 Sessions
7. GA4 Engaged Sessions

## Interpretation

The evaluation results suggest that machine learning can effectively prioritize pages for SEO optimization using historical search and engagement metrics. However, because the target variable was constructed using impressions and CTR, the reported performance should be interpreted carefully. The model is intended as a decision-support tool rather than an automated decision-making system.

## 5. Limitations

*What this work cannot claim.*

# Limitations & Honest Framing

## Limitations

Although the machine learning models achieved strong performance on the evaluation dataset, several limitations should be considered.

- The analysis was performed using historical Google Search Console (GSC) and Google Analytics (GA4) data from the FlyRank ML Internship dataset.
- The target variable was constructed using rule-based thresholds based on impressions and click-through rate (CTR). This may introduce label-derived leakage because some input features are closely related to the target.
- The evaluation was conducted on the available dataset and may not fully represent future website performance or unseen clients.
- Search engine algorithms, user behavior, and website content change over time, so model performance may decrease without periodic retraining.
- The recommendations produced by this project should be reviewed by SEO specialists before implementation and should not be used for fully automated decision-making.

## Honest Framing

The results presented in this study are **observed** on the evaluation dataset and should be interpreted as **decision-support** rather than proof of future performance. The model **suggests** which pages may benefit from SEO optimization, but it does not guarantee improved rankings or traffic. Additional validation on new data and continuous monitoring would strengthen confidence in the recommendations.

In [10]:
print("Leakage Audit Summary")
print("-" * 40)
print("✓ Historical features used")
print("✓ No future-window data used")
print("✓ Label-derived leakage reviewed")
print("✓ Results interpreted using honest claim language")
print("✓ Human review recommended before implementation")

Leakage Audit Summary
----------------------------------------
✓ Historical features used
✓ No future-window data used
✓ Label-derived leakage reviewed
✓ Results interpreted using honest claim language
✓ Human review recommended before implementation


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

# Ranked Recommendations

## Purpose

The ranked recommendations generated by this project are intended to help SEO analysts prioritize website pages that are most likely to benefit from optimization. The recommendations are based on historical Google Search Console (GSC) and Google Analytics (GA4) metrics and should be used as decision-support rather than automated actions.

## Action Playbook

| Priority | Reason Code | Recommended Action | Expected Benefit |
|----------|-------------|--------------------|------------------|
| High | HIGH_IMPRESSIONS_LOW_CTR | Improve page title and meta description | Increase Click-Through Rate (CTR) |
| High | LOW_SEARCH_POSITION | Improve on-page SEO and content quality | Improve search ranking |
| Medium | LOW_PAGE_ENGAGEMENT | Enhance page content and user experience | Increase user engagement |
| Medium | LOW_ORGANIC_SESSIONS | Strengthen internal linking and content promotion | Increase organic traffic |
| Low | MONITOR | Continue monitoring page performance | Maintain current performance |

## Intended Use

The recommendations should assist SEO professionals in prioritizing optimization efforts. They are designed to support human decision-making rather than replace expert judgment.

## Human Review

Before implementing any recommendation, an SEO specialist should review:

- Content quality
- Search intent
- Business objectives
- Brand guidelines
- Seasonal trends
- Current website strategy

## What Should NOT Be Automated

The following actions should always require human approval:

- Publishing new content
- Automatically changing page titles or meta descriptions
- Deleting or redirecting pages
- Business-critical SEO decisions
- Legal or compliance-related content changes

## Cost–Value Considerations

Pages with high impressions and low CTR should receive the highest priority because they offer the greatest potential improvement with relatively low implementation effort. Lower-priority pages should be monitored until additional data suggests that optimization is beneficial.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

# Artifacts the Paper Embeds

## Supporting Artifacts

The following artifacts were generated during this project and support the findings presented in this research paper:

### 1. Model Comparison
Comparison of Logistic Regression, Decision Tree, and Random Forest using Accuracy, Precision, Recall, and F1 Score.

### 2. Feature Importance
Feature importance generated from the Random Forest model to identify the most influential variables affecting SEO optimization recommendations.

### 3. Ranked Action Queue
A prioritized list of pages requiring SEO optimization, including recommendation reason codes and suggested actions.

### 4. Baseline Comparison
Comparison between the rule-based baseline and machine learning models to evaluate improvement in predictive performance.

These artifacts improve the transparency and reproducibility of the research while providing practical decision-support for SEO analysts.